# nano Beatrix — the binding's capacity, lane 1 of 3

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_capacity_lane1.ipynb)

**This notebook** is the read of [`nano_capacity.ipynb`](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_capacity.ipynb) - the question, the quantities, the decision block and the code are that notebook's - run as one of 3 lanes on 3 CPU runtimes at once. Every lane walks the thirty models in the same order. Before each model it reads the Hub at one commit: a model with its record there is done; a model with another lane's claim younger than 40 minutes is taken; the first model with neither is claimed by a small file under `bed/preflight/capacity_claims/`, written only if the repo has not moved since that read (else the read repeats), then read, and its record ships to `bed/reports/capacity/` as it lands. A lane ends when no model is free. A record does not depend on its lane: every model reads the same rows, bank and draws whichever lane reads it.

**The cells**: C1 pins the trainer (the read's version of record) and holds the decision block and the lane's name; C2 writes the held-out documents (the rows' filler), checks every model's weights on the Hub and probes the push path; C3 is the lane; C4 renders the tables of the records this lane read.

**Before C1**: Runtime > Change runtime type > CPU. The notebook's `HF_TOKEN` secret (write access to [AbstractPhil/alephllm-nano-beatrix-training](https://huggingface.co/AbstractPhil/alephllm-nano-beatrix-training)) with notebook access on: a lane claims before it reads, so a tokenless lane reads nothing. Run C1, then Runtime > Restart session and run all. A file named `STOP` in `/content/out` starts no new model; a restarted session takes back its own unfinished claim. The code and its tests: [AbstractEyes/geolip-nano-beatrix-trainer](https://github.com/AbstractEyes/geolip-nano-beatrix-trainer) (`geolip/nano_beatrix/capacity.py`, `tests/test_capacity.py`).

In [ ]:
# C1 — pinned installs + THE DECISION BLOCK (a CPU runtime). RESTART the session after the first run of this cell (Runtime > Restart session and run all).
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.85" "amoe-lora @ git+https://github.com/AbstractEyes/amoe-lora@v0.2.12"
import torch, amoe
import geolip.nano_beatrix as N
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 85), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart session, then rerun from C1')
assert amoe.__version__ == '0.2.12', 'the arm module of record is amoe-lora 0.2.12'
torch.set_num_threads(os.cpu_count() or 2)
print('geolip.nano_beatrix', N.__version__, '| amoe', amoe.__version__, '| torch', torch.__version__, '| cpu threads', torch.get_num_threads(),
      '| card', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (the read runs on the CPU)')

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN            # the runner and the shipper read it from the environment; the value is never printed

# ============ THE DECISION BLOCK — every setting from the plan, with its source beside it ============
OUT = '/content/out'                                   # the session folder: the held-out documents, the records, a model's weights while it is read
REPO = 'AbstractPhil/alephllm-nano-beatrix-training'  # the models' weights are read from here; every record ships here; tokenless = nothing ships
DEST = 'bed/reports/capacity'                          # the records' folder on the Hub
R = 192                                                # rows a model, half key-hard (the plan: 192 unless a model prices over 12 min; the CPU dry run: 0.56 s a row)
SEED = 1                                               # the rows' generator: the same rows for every model of a layout and keys' form
NS = (100, 1000, 10000, 100000, 1000000)               # the competitor counts (REAL: those the pool holds; EMULATED: all, the bank 10^6)
DRAWS = 8                                              # REAL draws an entry at each N
TAIL = '_L24000_ctx4096_frdyn_cap_mode_sz32-128-256-512_r4_n30_o0_askany_gag50'
FAM = {'LRK': '_cp4_gc48_kh50_va1_pt10b4h0_krt_ahdigit1_largerows',     # 48 records, the key-hard rows
       'LRP': '_cp4_gc48_va1_pt10b4h0_krt_ahdigit1_largerows',          # 48 records
       'LLK': '_cp4_gc48_kl_kh50_va1_pt10b4h0_krt_ahdigit1_largerows',  # 48 records, the long keys
       'KRK': '_cp4_kh50_va1_pt10b4h0_krt_ahdigit1_keyrot',             # 24 records, the key-hard rows
       'KRP': '_cp4_va1_pt10b4h0_krt_ahdigit1_keyrot'}                 # 24 records
LANE = 'lane1'                                      # this lane's name: its claims carry it
CORES = [(f'{f} s{s}', f'alhena-both_s{s}{TAIL}{FAM[f]}') for f in ('LRK', 'LRP', 'LLK', 'KRK', 'KRP') for s in range(1, 7)]
print('decision block:', LANE, '|', len(CORES), 'models |', R, 'rows a model, seed', SEED, '| N', NS, '|', DRAWS, 'draws | ships to',
      REPO + '/' + DEST if HF_TOKEN else 'nothing (tokenless)')

In [ ]:
# C2 (P) — PREFLIGHT: the held-out documents (the rows' filler), every model's weights on the Hub, the push probe. Nothing is read yet.
import json, os, time
from huggingface_hub import HfApi
from geolip.nano_beatrix import bed_corpus
HELD = bed_corpus.write_heldout(f'{OUT}/bed')         # the first 2,048 documents of fineweb-edu sample-10BT (the beds' held-out head)
print('held-out documents', HELD, os.path.getsize(HELD) / 1e6, 'MB')
api = HfApi(token=HF_TOKEN or False)
FILES = set(api.list_repo_files(REPO))
missing = [lab for lab, tag in CORES if f'bed/weights/{tag}/key_rot.safetensors' not in FILES]
assert not missing, f'models without their stored phases on the Hub: {missing}'
done_hub = [lab for lab, tag in CORES if f'{DEST}/{tag}_capacity_r{R}_s{SEED}.json' in FILES]
print('every model on the Hub with its phases:', len(CORES), '| records already shipped (done before):', done_hub or 'none')
PUSH_OK = False
if HF_TOKEN:
    path = f'bed/preflight/push_probe_capacity_{LANE}.json'
    body = json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'cpu_threads': torch.get_num_threads(),
                       'models': [lab for lab, _ in CORES], 'rows': R}).encode()
    api.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=REPO, commit_message=f'nano capacity: push probe ({LANE})')
    PUSH_OK = path in set(api.list_repo_files(REPO))
    print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)
else:
    print('TOKENLESS: no probe written; the cells below read and ship nothing')

In [ ]:
# C3 (R) — THE LANE: the thirty models in the single notebook's order, one at a time. Before each model the Hub is read at one commit: a
# model with its record there (or in OUT/capacity) is done; a model with another lane's claim younger than FRESH_S is taken; the first
# model with neither is claimed (a small file under CLAIMS, written only if the repo has not moved since that read - else the read repeats),
# read, and its record shipped as it lands (a background thread with retries; the local record is the done mark). A STOP file in OUT starts
# no new model. The lane ends when no model is free. The job's size, the time spent and the time left print after every model.
import json, os, random, shutil, threading, time
from huggingface_hub import snapshot_download
from geolip.nano_beatrix import capacity as cp, rotary_scale as rs
assert HF_TOKEN and PUSH_OK, 'a lane claims before it reads: it needs the token and the push probe read back (C2)'
held = rs.load_heldout(HELD)
REC = f'{OUT}/capacity'
CLAIMS = 'bed/preflight/capacity_claims'
FRESH_S = 40 * 60                                      # a claim this old with no record is a stopped lane's: the model is free again
os.makedirs(REC, exist_ok=True)
threads, failed = [], []

def _ship(path):
    for i in range(6):
        try:
            api.upload_file(path_or_fileobj=path, path_in_repo=f'{DEST}/{os.path.basename(path)}', repo_id=REPO,
                            commit_message=f'nano capacity: {os.path.basename(path)[:90]}')
            return
        except Exception as e:
            print('ship retry', i + 1, os.path.basename(path)[:60], type(e).__name__)
            time.sleep(min(300, 10 * 2 ** i))
    failed.append(path)

def ship(path):
    t = threading.Thread(target=_ship, args=(path,))
    t.start()
    threads.append(t)

def snap():
    # the repo's head commit and its files, read at that one commit
    sha = api.repo_info(REPO).sha
    return sha, set(api.list_repo_files(REPO, revision=sha))

def done(tag, files):
    name = cp.record_name(tag, R, SEED)
    return f'{DEST}/{name}' in files or os.path.exists(f'{REC}/{name}')

def claims(label, files):
    # the fresh claims on a model, [(lane, utc seconds)], from the claim files' names: <label>__<lane>__<utc seconds>.json
    pre = f"{CLAIMS}/{label.replace(' ', '_')}__"
    out = []
    for f in files:
        if f.startswith(pre) and f.endswith('.json'):
            lane, utc = f[len(pre):-len('.json')].split('__')
            if time.time() - int(utc) < FRESH_S:
                out.append((lane, int(utc)))
    return out

def pick():
    # -> (label, tag) claimed by this lane, or None when no model is free
    for attempt in range(40):
        sha, files = snap()
        for label, tag in CORES:                       # this lane's own claim on an unfinished model (a restarted session) is still its
            if not done(tag, files) and any(lane == LANE for lane, _ in claims(label, files)):
                return label, tag
        free = [(label, tag) for label, tag in CORES if not done(tag, files) and not claims(label, files)]
        if not free:
            return None
        label, tag = free[0]
        path = f"{CLAIMS}/{label.replace(' ', '_')}__{LANE}__{int(time.time())}.json"
        try:
            api.upload_file(path_or_fileobj=json.dumps({'lane': LANE, 'model': tag}).encode(), path_in_repo=path, repo_id=REPO,
                            commit_message=f'nano capacity: {LANE} claims {label}', parent_commit=sha)
            return label, tag
        except Exception as e:                          # the repo moved since the read (another lane's claim or record) or the call failed:
            print('claim retry', attempt + 1, label, type(e).__name__)   # read again (a claim that landed anyway is found as this lane's)
            time.sleep(random.uniform(3, 12))
    raise RuntimeError('no claim after 40 tries')

_, FILES = snap()
for f in sorted(os.listdir(REC)):                     # local records not on the Hub: shipped first
    if f.endswith('.json') and f'{DEST}/{f}' not in FILES:
        ship(f'{REC}/{f}')
t0, read_n, recs = time.time(), 0, {}
while True:
    if os.path.exists(f'{OUT}/STOP'):
        print('STOP present: no new model starts')
        break
    got = pick()
    if got is None:
        print('no model is free: the lane ends')
        break
    label, tag = got
    local = f'{REC}/{cp.record_name(tag, R, SEED)}'
    t1 = time.time()
    snapshot_download(REPO, allow_patterns=[f'bed/weights/{tag}/*'], local_dir=f'{OUT}/w', token=HF_TOKEN)
    cp.run(f'{OUT}/w/bed/weights/{tag}', held, REC, R=R, seed=SEED, ns=NS, draws=DRAWS, log=print)
    ship(local)
    shutil.rmtree(f'{OUT}/w/bed/weights/{tag}', ignore_errors=True)
    read_n += 1
    rec = recs[label] = json.load(open(local, encoding='utf-8'))
    e = rec['emulated']
    _, files = snap()
    open_n = sum(not done(tag_, files) for _, tag_ in CORES)
    spent = time.time() - t0
    print(f"{label} [{LANE}]: s {rec['s_core']:.2f} | mean leak 10^4 {e['10000']['mean']:.1e} (moment {e['10000']['moment']:.1e}), "
          f"10^6 {e['1000000']['mean']:.2f} (moment {e['1000000']['moment']:.2f}) | look error 10^4 {e['10000']['look_error']:.4f}, "
          f"10^6 {e['1000000']['look_error']:.4f} | {time.time() - t1:.0f} s | this lane: {read_n} read in {spent / 60:.1f} min | "
          f"{open_n} of {len(CORES)} models without a record, ~{open_n * spent / read_n / 3 / 60:.0f} min left at 3 lanes")
for t in threads:
    t.join()
_, FILES = snap()
print('this lane read', read_n, '| UNSHIPPED:', [f for f in sorted(os.listdir(REC)) if f.endswith('.json') and f'{DEST}/{f}' not in FILES])

In [ ]:
# C4 — THE TABLES (the records in OUT/capacity): per model the margin s, the mean (median) leak EMULATED at each N against the moment,
# the look error, and the REAL means where the pool holds N.
from IPython.display import Markdown, display
rows = ['| model | s | N = 10^3: mean (median) / moment | 10^4 | 10^5 | 10^6 | look error 10^4 / 10^6 | REAL mean 10^3 / 10^4 |',
        '|---|---|---|---|---|---|---|---|']
for label, tag in CORES:
    if label not in recs:
        continue
    r = recs[label]
    e = r['emulated']
    f = lambda n: f"{e[n]['mean']:.1e} ({e[n]['median']:.1e}) / {e[n]['moment']:.1e}"
    rr = ' / '.join(f"{r['real'][n]['mean']:.1e}" if n in r['real'] else '-' for n in ('1000', '10000'))
    rows.append(f"| {label} | {r['s_core']:.2f} | {f('1000')} | {f('10000')} | {f('100000')} | {f('1000000')} | "
                f"{e['10000']['look_error']:.4f} / {e['1000000']['look_error']:.4f} | {rr} |")
display(Markdown('\n'.join(rows)))